In [17]:
import numpy as np
import pandas as pd

In [18]:
df_full = pd.read_csv('cox-violent-parsed_filt.csv')
df_full.head()

,id,name,first,last,sex,dob,age,age_cat,race,juv_fel_count,...,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,priors_count.1,event
0,1.0,miguel hernandez,miguel,hernandez,Male,18/04/1947,69,Greater than 45,Other,0,...,NaN,Risk of Recidivism,1,Low,14/08/2013,Risk of Violence,1,Low,0,0
1,2.0,miguel hernandez,miguel,hernandez,Male,18/04/1947,69,Greater than 45,Other,0,...,NaN,Risk of Recidivism,1,Low,14/08/2013,Risk of Violence,1,Low,0,0
2,3.0,michael ryan,michael,ryan,Male,06/02/1985,31,25 - 45,Caucasian,0,...,NaN,Risk of Recidivism,5,Medium,31/12/2014,Risk of Violence,2,Low,0,0
3,4.0,kevon dixon,kevon,dixon,Male,22/01/1982,34,25 - 45,African-American,0,...,Felony Battery (Dom Strang),Risk of Recidivism,3,Low,27/01/2013,Risk of Violence,1,Low,0,1
4,5.0,ed philo,ed,philo,Male,14/05/1991,24,Less than 25,African-American,0,...,NaN,Risk of Recidivism,4,Low,14/04/2013,Risk of Violence,3,Low,4,0


In [19]:
df = df_full[["age", "age_cat", "juv_fel_count", "juv_misd_count", "juv_other_count", "priors_count", "c_charge_degree", "c_days_from_compas", "is_recid"]].copy()
df.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,c_days_from_compas,is_recid
0,69,Greater than 45,0,0,0,0,(F3),1.0,0
1,69,Greater than 45,0,0,0,0,(F3),1.0,0
2,31,25 - 45,0,0,0,0,NaN,NaN,-1
3,34,25 - 45,0,0,0,0,(F3),1.0,1
4,24,Less than 25,0,0,1,4,(F3),1.0,1


In [20]:
df['age_cat'] = df['age_cat'].str.replace('Greater than ', '')
df['age_cat'] = df['age_cat'].str.replace('Less than ', '')
for i in range (0, df.shape[0]):
    if df.iloc[i, 1] == '45':
        df.iloc[i, 1] = 'Senior'
    elif df.iloc[i, 1] == '25':
        df.iloc[i, 1] = 'Young'
    else: 
        df.iloc[i, 1] = 'Adult'
df.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,c_days_from_compas,is_recid
0,69,Senior,0,0,0,0,(F3),1.0,0
1,69,Senior,0,0,0,0,(F3),1.0,0
2,31,Adult,0,0,0,0,NaN,NaN,-1
3,34,Adult,0,0,0,0,(F3),1.0,1
4,24,Young,0,0,1,4,(F3),1.0,1


In [21]:
df.isnull().sum()

age                     0
age_cat                 0
juv_fel_count           0
juv_misd_count          0
juv_other_count         0
priors_count            0
c_charge_degree       867
c_days_from_compas    867
is_recid                0
dtype: int64

In [22]:
df = df.dropna(subset=['c_charge_degree', 'c_days_from_compas'])
df['c_days_from_compas'] = df['c_days_from_compas'].astype(int)
df.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,c_days_from_compas,is_recid
0,69,Senior,0,0,0,0,(F3),1,0
1,69,Senior,0,0,0,0,(F3),1,0
3,34,Adult,0,0,0,0,(F3),1,1
4,24,Young,0,0,1,4,(F3),1,1
5,24,Young,0,0,1,4,(F3),1,1


In [23]:
severity_map = {
    '(F1)': 13, '(F2)': 12, '(F3)': 11,
    '(F5)': 10, '(F6)': 9,  '(F7)': 8,
    '(TCX)': 7, '(M1)': 6,  '(M2)': 5,
    '(MO3)': 4, '(CO3)': 3, '(CT)': 2,
    '(NI0)': 1, '(X)': 0
}

df['c_charge_degree'] = df['c_charge_degree'].map(severity_map)       
df.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,c_days_from_compas,is_recid
0,69,Senior,0,0,0,0,11,1,0
1,69,Senior,0,0,0,0,11,1,0
3,34,Adult,0,0,0,0,11,1,1
4,24,Young,0,0,1,4,11,1,1
5,24,Young,0,0,1,4,11,1,1


In [24]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
num_cols = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count','c_charge_degree', 'c_days_from_compas']

encoder = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(), ['age_cat'])
])
encoder

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [25]:
from sklearn.model_selection import train_test_split
df = df[df['is_recid'] != -1]
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.head()

,age,age_cat,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,c_days_from_compas
10563,25,Adult,0,0,0,9,6,1
15181,37,Adult,0,0,0,2,11,1
7054,35,Adult,0,0,1,0,11,1
9707,25,Adult,0,1,0,2,11,1
4635,23,Young,0,0,1,4,11,1


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
classifier = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
model = Pipeline([
    ('encoder', encoder),
    ('classifier', classifier)
])
model

,steps,"[('encoder', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [27]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [28]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(accuracy_score(y_test, y_pred))
cm

0.7893982808022922


array([[1454,  361],
       [ 374, 1301]])

In [29]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80      1815
           1       0.78      0.78      0.78      1675

    accuracy                           0.79      3490
   macro avg       0.79      0.79      0.79      3490
weighted avg       0.79      0.79      0.79      3490



In [30]:
from sklearn.metrics import accuracy_score, confusion_matrix

X_test_fair = X_test.copy()
X_test_fair['race'] = df_full.loc[X_test.index, 'race']

for group in X_test_fair['race'].unique():
    mask = X_test_fair['race'] == group
    y_true_group = y_test[mask]
    y_pred_group = y_pred[mask]
    
    tn, fp, fn, tp = confusion_matrix(y_true_group, y_pred_group).ravel()
    
    print(f"\n{group}")
    print(f"  Accuracy:           {accuracy_score(y_true_group, y_pred_group):.2f}")
    print(f"  False Positive Rate:{fp / (fp + tn):.2f}")
    print(f"  False Negative Rate:{fn / (fn + tp):.2f}")
    print(f"  Sample size:        {mask.sum()}")


Hispanic
  Accuracy:           0.78
  False Positive Rate:0.20
  False Negative Rate:0.27
  Sample size:        287

Caucasian
  Accuracy:           0.79
  False Positive Rate:0.16
  False Negative Rate:0.28
  Sample size:        1142

African-American
  Accuracy:           0.79
  False Positive Rate:0.24
  False Negative Rate:0.19
  Sample size:        1894

Other
  Accuracy:           0.79
  False Positive Rate:0.12
  False Negative Rate:0.37
  Sample size:        145

Asian
  Accuracy:           1.00
  False Positive Rate:0.00
  False Negative Rate:0.00
  Sample size:        15

Native American
  Accuracy:           0.86
  False Positive Rate:0.25
  False Negative Rate:0.00
  Sample size:        7


In [31]:
import joblib
joblib.dump(model, 'model.pkl')

['model.pkl']